# 01 - Dataset Collection

## AI Emotion RAG Assistant

This notebook loads the GoEmotions dataset exported from Hugging Face,
verifies dataset quality and creates metadata for the remaining notebooks.


In [26]:
import warnings
warnings.filterwarnings("ignore")

import re
import json
from pathlib import Path

import pandas as pd


In [27]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT/"datasets"/"raw"/"emotion"
META_DIR = PROJECT_ROOT/"datasets"/"metadata"
TABLE_DIR = PROJECT_ROOT/"outputs"/"tables"

META_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = RAW_DIR/"train.csv"
VALID_PATH = RAW_DIR/"validation.csv"
TEST_PATH = RAW_DIR/"test.csv"


## Helper Function

In [28]:
def parse_labels(x):
    if pd.isna(x):
        return []
    return [int(i) for i in re.findall(r"\d+", str(x))]


## Load Dataset

In [29]:
train_df=pd.read_csv(TRAIN_PATH)
valid_df=pd.read_csv(VALID_PATH)
test_df=pd.read_csv(TEST_PATH)

for data in (train_df,valid_df,test_df):
    data["labels"]=data["labels"].apply(parse_labels)

print(f"Train      : {len(train_df):,}")
print(f"Validation : {len(valid_df):,}")
print(f"Test       : {len(test_df):,}")

display(train_df.head())


Train      : 43,410
Validation : 5,426
Test       : 5,427


,text,labels,id
0,My favourite food is anything I didn't have to...,[27],eebbqej
1,"Now if he does off himself, everyone will thin...",[27],ed00q6i
2,WHY THE FUCK IS BAYLESS ISOING,[2],eezlygj
3,To make her feel threatened,[14],ed7ypvh
4,Dirty Southern Wankers,[3],ed0bdzj


## Dataset Structure

In [30]:
for name,data in {
    "Train":train_df,
    "Validation":valid_df,
    "Test":test_df
}.items():
    print("="*60)
    print(name)
    print(data.shape)
    display(data.head(3))


Train
(43410, 3)


,text,labels,id
0,My favourite food is anything I didn't have to...,[27],eebbqej
1,"Now if he does off himself, everyone will thin...",[27],ed00q6i
2,WHY THE FUCK IS BAYLESS ISOING,[2],eezlygj


Validation
(5426, 3)


,text,labels,id
0,Is this in New Orleans?? I really feel like th...,[27],edgurhb
1,"You know the answer man, you are programmed to...","[4, 27]",ee84bjg
2,I've never been this sad in my life!,[25],edcu99z


Test
(5427, 3)


,text,labels,id
0,I’m really sorry about your situation :( Altho...,[25],eecwqtt
1,It's wonderful because it's awful. At not with.,[0],ed5f85d
2,"Kings fan here, good luck to you guys! Will be...",[13],een27c3


## Merge Dataset

In [31]:
df=pd.concat([train_df,valid_df,test_df],ignore_index=True)
print(df.shape)


(54263, 3)


## Data Quality

In [32]:
print("Missing Values")
display(df.isnull().sum())

for name,data in {
    "Train":train_df,
    "Validation":valid_df,
    "Test":test_df
}.items():
    print("\n"+"="*50)
    print(name)
    print("Duplicate IDs :",data["id"].duplicated().sum())
    print("Duplicate Text:",data["text"].duplicated().sum())


Missing Values


text      0
labels    0
id        0
dtype: int64


Train
Duplicate IDs : 0
Duplicate Text: 183

Validation
Duplicate IDs : 0
Duplicate Text: 3

Test
Duplicate IDs : 0
Duplicate Text: 6


## Label Statistics

In [33]:
df["num_labels"]=df["labels"].apply(len)

display(df["num_labels"].describe())

display(df["num_labels"].value_counts().sort_index())


count    54263.000000
mean         1.175976
std          0.416492
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          5.000000
Name: num_labels, dtype: float64

num_labels
1    45446
2     8124
3      655
4       37
5        1
Name: count, dtype: int64

## Emotion Mapping

In [34]:
emotion_names=[
"admiration","amusement","anger","annoyance","approval",
"caring","confusion","curiosity","desire","disappointment",
"disapproval","disgust","embarrassment","excitement","fear",
"gratitude","grief","joy","love","nervousness",
"optimism","pride","realization","relief","remorse",
"sadness","surprise","neutral"
]

emotion_df=pd.DataFrame({
    "label_id":range(28),
    "emotion":emotion_names
})

display(emotion_df)

emotion_df.to_csv(META_DIR/"emotion_labels.csv",index=False)

with open(META_DIR/"label_mapping.json","w") as f:
    json.dump({i:n for i,n in enumerate(emotion_names)},f,indent=4)


,label_id,emotion
0,0,admiration
1,1,amusement
2,2,anger
3,3,annoyance
4,4,approval
5,5,caring
6,6,confusion
7,7,curiosity
8,8,desire
9,9,disappointment


## Export Metadata

In [35]:
dataset_info={
"train_samples":len(train_df),
"validation_samples":len(valid_df),
"test_samples":len(test_df),
"total_samples":len(df),
"num_emotions":28,
"avg_labels_per_sample":round(df["num_labels"].mean(),3),
"max_labels_per_sample":int(df["num_labels"].max())
}

with open(META_DIR/"dataset_info.json","w") as f:
    json.dump(dataset_info,f,indent=4)

summary=pd.DataFrame(dataset_info.items(),columns=["Metric","Value"])
summary.to_csv(TABLE_DIR/"dataset_summary.csv",index=False)
display(summary)


,Metric,Value
0,train_samples,43410.000
1,validation_samples,5426.000
2,test_samples,5427.000
3,total_samples,54263.000
4,num_emotions,28.000
5,avg_labels_per_sample,1.176
6,max_labels_per_sample,5.000


# Conclusion

Completed

- Loaded train / validation / test
- Parsed labels
- Verified data quality
- Analysed label statistics
- Generated metadata

Next notebook: **02_Exploratory_Data_Analysis.ipynb**
